In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install selenium webdriver-manager pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [26]:
#300 verinin yorumlarını çeken kod
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

import pandas as pd
import time
import re
import os
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse


# ============================================================
# AYARLAR
# ============================================================

MAX_YORUM = 20
EXCEL_DOSYASI = "ham_laptop_verileri.xlsx"
CIKTI_DOSYASI = "ham_laptop_yorumlari.xlsx"
ARA_KAYIT_ARALIGI = 10

PROFILE_PATH = os.path.abspath("selenium_chrome_profile")


# ============================================================
# EXCEL OKU
# ============================================================

df_urunler = pd.read_excel(EXCEL_DOSYASI)

print("Ürün dosyası okundu.")
print("Toplam ürün:", len(df_urunler))

if "ürün_linki" not in df_urunler.columns:
    raise Exception("Excel dosyasında 'ürün_linki' sütunu bulunamadı.")

if "ürün_adı" not in df_urunler.columns:
    raise Exception("Excel dosyasında 'ürün_adı' sütunu bulunamadı.")


# ============================================================
# DAHA ÖNCE ÇEKİLENLER VARSA OKU
# ============================================================

if os.path.exists(CIKTI_DOSYASI):
    eski_df = pd.read_excel(CIKTI_DOSYASI)
    yorumlar = eski_df.to_dict("records")
    tamamlanan_urunler = set(eski_df["ürün_id"].dropna().astype(int).unique())
    print("Mevcut yorum dosyası bulundu.")
    print("Daha önce tamamlanan ürün sayısı:", len(tamamlanan_urunler))
else:
    yorumlar = []
    tamamlanan_urunler = set()


# ============================================================
# CHROME BAŞLAT
# ============================================================

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument(f"--user-data-dir={PROFILE_PATH}")
options.add_argument("--profile-directory=Default")

options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.execute_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)

wait = WebDriverWait(driver, 20)


# ============================================================
# FONKSİYONLAR
# ============================================================

def bekle(saniye=2):
    time.sleep(saniye)


def temizle(text):
    text = str(text).strip()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def gercek_yorum_mu(text):
    if not text:
        return False

    if len(text) < 35:
        return False

    yasakli = [
        "Değerlendirilen özellikler",
        "İşlemci",
        "Hafıza",
        "Görüntü Kalitesi",
        "Batarya",
        "Bu değerlendirme faydalı mı",
        "Yorumu faydalı buldunuz mu",
        "Kullanıcı bu ürünü",
        "Evet",
        "Hayır",
        "Filtrele",
        "Sırala",
        "Önerilen sıralama",
        "En yeni",
        "En eski"
    ]

    return not any(k.lower() in text.lower() for k in yasakli)


def urun_yorum_linki_mi(href):
    if not href:
        return False

    href_lower = href.lower()

    if "/yorumlarim/" in href_lower:
        return False

    if "yorum-bekleyenler" in href_lower:
        return False

    if "yorumlarim" in href_lower:
        return False

    if "-yorumlari" not in href_lower:
        return False

    if "-p-" not in href_lower:
        return False

    return True


def yorum_url_sayfa_yap(base_url, sayfa_no):
    parsed = urlparse(base_url)
    query = parse_qs(parsed.query)

    query["sayfa"] = [str(sayfa_no)]
    new_query = urlencode(query, doseq=True)

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        new_query,
        parsed.fragment
    ))


def yorum_sayfasi_linkini_bul():
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
        bekle(2)

        try:
            review_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//button[contains(@aria-label,'Değerlendirme')]")
                )
            )
            driver.execute_script("arguments[0].click();", review_btn)
            bekle(3)
        except Exception:
            pass

        linkler = driver.find_elements(By.TAG_NAME, "a")
        aday_linkler = []

        for link in linkler:
            href = link.get_attribute("href") or ""

            if urun_yorum_linki_mi(href):
                aday_linkler.append(href)

        if aday_linkler:
            return aday_linkler[0]

    except Exception:
        pass

    return None


def yorum_spanlarini_bul():
    try:
        spanlar = driver.find_elements(
            By.CSS_SELECTOR,
            "span[style='text-align:start']"
        )

        if len(spanlar) > 0:
            return spanlar
    except Exception:
        pass

    try:
        return driver.find_elements(
            By.CSS_SELECTOR,
            "div[class*='ReviewCard'] span"
        )
    except Exception:
        return []


def kart_bul(span):
    try:
        return span.find_element(
            By.XPATH,
            "./ancestor::div[contains(@class,'ReviewCard')]"
        )
    except Exception:
        return None


def tarih_bul(span):
    try:
        kart = kart_bul(span)

        if kart is None:
            return "Bulunamadı"

        text = kart.text

        kaliplar = [
            r"\d{1,2}\s+[A-Za-zÇĞİÖŞÜçğıöşü]+,\s*[A-Za-zÇĞİÖŞÜçğıöşü]+",
            r"\d{1,2}\s+[A-Za-zÇĞİÖŞÜçğıöşü]+\s+\d{4}",
            r"\d{1,2}\.\d{1,2}\.\d{4}",
            r"\d{4}-\d{2}-\d{2}"
        ]

        for kalip in kaliplar:
            eslesme = re.search(kalip, text)
            if eslesme:
                return eslesme.group(0)

    except Exception:
        pass

    return "Bulunamadı"


def yildiz_bul(span):
    try:
        kart = kart_bul(span)

        if kart is None:
            return "Bulunamadı"

        aria_elemanlar = kart.find_elements(By.XPATH, ".//*[@aria-label]")

        for el in aria_elemanlar:
            aria = el.get_attribute("aria-label") or ""

            eslesme = re.search(
                r"(\d,[0-9]|\d)\s*yıldız",
                aria,
                re.IGNORECASE
            )

            if eslesme:
                return eslesme.group(1)

    except Exception:
        pass

    return "Bulunamadı"


def yorum_sayfasi_yuklendi_mi(timeout=15):
    baslangic = time.time()

    while time.time() - baslangic < timeout:
        spanlar = yorum_spanlarini_bul()

        if len(spanlar) > 0:
            return True

        bekle(1)

    return False


def sayfadaki_yorumlari_cek(urun_id, urun_adi, urun_linki, gorulen_yorumlar):
    yorum_sayfasi_yuklendi_mi(timeout=15)

    spanlar = yorum_spanlarini_bul()

    print("Bu sayfada bulunan span sayısı:", len(spanlar))

    yeni_sayi = 0

    for span in spanlar:
        try:
            yorum_metni = temizle(span.get_attribute("innerText") or span.text)
        except StaleElementReferenceException:
            continue

        if not gercek_yorum_mu(yorum_metni):
            continue

        kontrol = yorum_metni.lower()

        if kontrol in gorulen_yorumlar:
            continue

        gorulen_yorumlar.add(kontrol)

        yorumlar.append({
            "ürün_id": urun_id,
            "ürün_adı": urun_adi,
            "yıldız": yildiz_bul(span),
            "yorum_metni": yorum_metni,
            "yorum_tarihi": tarih_bul(span),
            "ürün_linki": urun_linki,
            "yorum_sayfa_url": driver.current_url
        })

        yeni_sayi += 1

        if yeni_sayi >= MAX_YORUM:
            break

    return yeni_sayi


def kaydet():
    df_yorumlar = pd.DataFrame(yorumlar)
    df_yorumlar.to_excel(CIKTI_DOSYASI, index=False)
    print("Ara kayıt yapıldı. Toplam yorum satırı:", len(df_yorumlar))


# ============================================================
# ANA AKIŞ
# ============================================================

try:
    islenen_urun_sayisi = 0

    for index, row in df_urunler.iterrows():
        urun_id = index + 1

        if urun_id in tamamlanan_urunler:
            print(f"{urun_id}. ürün daha önce tamamlanmış, geçiliyor.")
            continue

        urun_adi = str(row["ürün_adı"])
        urun_linki = str(row["ürün_linki"])

        print("\n==================================================")
        print(f"{urun_id}. ürün işleniyor")
        print(urun_adi)
        print("==================================================")

        try:
            driver.get(urun_linki)
            bekle(5)

            if "login" in driver.current_url.lower() or "giris" in driver.current_url.lower():
                print("Giriş sayfasına yönlendirildi. Bu ürün atlanıyor.")
                continue

            yorum_base_url = yorum_sayfasi_linkini_bul()

            if not yorum_base_url:
                print("Yorum linki bulunamadı. Ürün atlandı.")
                continue

            print("Yorum sayfası:")
            print(yorum_base_url)

            gorulen_yorumlar = set()
            urun_yorum_baslangic = len(yorumlar)

            for sayfa_no in range(1, 15):
                urun_icin_cekilen = len(yorumlar) - urun_yorum_baslangic

                if urun_icin_cekilen >= MAX_YORUM:
                    break

                yorum_url = yorum_url_sayfa_yap(yorum_base_url, sayfa_no)

                print(f"{sayfa_no}. yorum sayfası açılıyor...")
                driver.get(yorum_url)
                bekle(4)

                if "/yorumlarim/" in driver.current_url.lower():
                    print("Yanlış yorumlarım sayfasına gidildi. Ürün atlanıyor.")
                    break

                if "login" in driver.current_url.lower() or "giris" in driver.current_url.lower():
                    print("Yorum sayfasında giriş ekranına yönlendirdi. Ürün atlanıyor.")
                    break

                onceki_sayi = len(yorumlar)

                yeni_sayi = sayfadaki_yorumlari_cek(
                    urun_id,
                    urun_adi,
                    urun_linki,
                    gorulen_yorumlar
                )

                urun_icin_toplam = len(yorumlar) - urun_yorum_baslangic

                print(f"{sayfa_no}. sayfadan yeni yorum: {yeni_sayi}")
                print(f"Bu ürün için toplam yorum: {urun_icin_toplam}/{MAX_YORUM}")

                if yeni_sayi == 0:
                    print("Yeni yorum bulunamadı. Sonraki ürüne geçiliyor.")
                    break

                if len(yorumlar) == onceki_sayi:
                    print("Yorum sayısı değişmedi. Sonraki ürüne geçiliyor.")
                    break

            islenen_urun_sayisi += 1

            if islenen_urun_sayisi % ARA_KAYIT_ARALIGI == 0:
                kaydet()

        except Exception as e:
            print(f"{urun_id}. üründe hata oluştu:", e)
            kaydet()
            continue

finally:
    driver.quit()
    kaydet()


print("\nİşlem tamamlandı.")
print("Çıktı dosyası:", CIKTI_DOSYASI)
print("Toplam yorum satırı:", len(yorumlar))

Ürün dosyası okundu.
Toplam ürün: 300

1. ürün işleniyor
Lenovo Ideapad Slim 3 AMD Ryzen 7 7735HS 16GB 512GB SSD Freedos 15.3" Taşınabilir Bilgisayar 83K70098TR
Yorum sayfası:
https://www.hepsiburada.com/lenovo-ideapad-slim-3-amd-ryzen-7-7735hs-16gb-512gb-ssd-freedos-15-3-tasinabilir-bilgisayar-83k70098tr-p-HBCV0000CM0YNQ-yorumlari
1. yorum sayfası açılıyor...
Bu sayfada bulunan span sayısı: 20
1. sayfadan yeni yorum: 8
Bu ürün için toplam yorum: 8/20
2. yorum sayfası açılıyor...
Bu sayfada bulunan span sayısı: 10
2. sayfadan yeni yorum: 8
Bu ürün için toplam yorum: 16/20
3. yorum sayfası açılıyor...
Bu sayfada bulunan span sayısı: 10
3. sayfadan yeni yorum: 9
Bu ürün için toplam yorum: 25/20

2. ürün işleniyor
Lenovo Ideapad Slim 3 AMD Ryzen 5 7535HS 16GB 512GB SSD Freedos 15.3" Taşınabilir Bilgisayar 83K7009ETR
Yorum sayfası:
https://www.hepsiburada.com/lenovo-ideapad-slim-3-amd-ryzen-5-7535hs-16gb-512gb-ssd-freedos-15-3-tasinabilir-bilgisayar-83k7009etr-p-HBCV0000D9H5PV-yorumlari
1.